In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

print("✅ Библиотеки загружены")

In [ ]:
print("Загрузка данных Fashion MNIST...")
data = pd.read_csv('fashion_mnist.csv')

print(f"Размер данных: {data.shape}")
print(f"Количество классов: {data['label'].nunique()}")
print(f"Метки классов: {sorted(data['label'].unique())}")

plt.figure(figsize=(10, 6))
data['label'].value_counts().sort_index().plot(kind='bar', color='steelblue')
plt.title('Распределение классов в датасете')
plt.xlabel('Класс')
plt.ylabel('Количество')
plt.xticks(range(10), [str(i) for i in range(10)])
plt.grid(alpha=0.3)
plt.show()

In [ ]:
X = data.drop('label', axis=1).values
y = data['label'].values

X = X / 255.0

print(f"Признаки: {X.shape}")
print(f"Целевая переменная: {y.shape}")
print(f"Диапазон значений пикселей: [{X.min():.2f}, {X.max():.2f}]")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nОбучающая выборка: {X_train.shape}")
print(f"Тестовая выборка: {X_test.shape}")

In [ ]:
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

fig, axes = plt.subplots(2, 5, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    idx = np.where(y == i)[0][0]
    img = X[idx].reshape(28, 28)
    ax.imshow(img, cmap='gray')
    ax.set_title(f'{class_names[i]} ({i})')
    ax.axis('off')
plt.suptitle('Примеры изображений из Fashion MNIST', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
print("Обучение модели Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\nТочность модели на тестовой выборке: {accuracy:.4f} ({accuracy*100:.2f}%)")

print("\nОтчет классификации по каждому классу:")
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Матрица ошибок классификации', fontsize=14)
plt.xlabel('Предсказанный класс', fontsize=12)
plt.ylabel('Истинный класс', fontsize=12)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
errors = (y_pred != y_test)
error_indices = np.where(errors)[0]

print(f"Всего ошибок: {len(error_indices)} ({(len(error_indices)/len(y_test)*100):.2f}%)")
print("\nТоп-5 самых частых ошибок:")

from collections import Counter
error_pairs = [(y_test[i], y_pred[i]) for i in error_indices[:100]]
common_errors = Counter(error_pairs).most_common(5)

for (true, pred), count in common_errors:
    print(f"  {class_names[true]} → {class_names[pred]}: {count} раз")

fig, axes = plt.subplots(2, 5, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i < len(error_indices):
        idx = error_indices[i]
        img = X_test[idx].reshape(28, 28)
        ax.imshow(img, cmap='gray')
        ax.set_title(f'True: {class_names[y_test[idx]]}\nPred: {class_names[y_pred[idx]]}')
        ax.axis('off')
plt.suptitle('Примеры ошибочных классификаций', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
importances = rf_model.feature_importances_
importance_img = importances.reshape(28, 28)

plt.figure(figsize=(10, 8))
plt.imshow(importance_img, cmap='hot', interpolation='nearest')
plt.colorbar(label='Важность')
plt.title('Тепловая карта важности пикселей для модели', fontsize=14)
plt.xlabel('Ширина (пиксели)')
plt.ylabel('Высота (пиксели)')
plt.show()

top_indices = np.argsort(importances)[-10:][::-1]
print("\nТоп-10 самых важных позиций пикселей:")
for i, idx in enumerate(top_indices):
    row, col = idx // 28, idx % 28
    print(f"  {i+1}. Позиция ({row}, {col}): важность = {importances[idx]:.6f}")

In [ ]:
from sklearn.linear_model import LogisticRegression

print("Сравнение с логистической регрессией...")
lr_model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)
accuracy_lr = accuracy_score(y_test, y_pred_lr)

print(f"\nСравнение моделей:")
print(f"  Random Forest: {accuracy:.4f}")
print(f"  Logistic Regression: {accuracy_lr:.4f}")
print(f"  Разница: {accuracy - accuracy_lr:.4f} в пользу Random Forest")